# Financial Inclusion in Africa

**Team:** Andi, Muhammet  
**Ziel:** Vorhersagen, ob eine Person ein Bankkonto hat (`bank_account`: Yes / No).  
**Daten:** Zindi, Umfragen aus Kenia, Ruanda, Tansania und Uganda (2016–2018).

## Milestone 0: Daten laden

In [ ]:
import pandas as pd  # Werkzeugkasten pandas holen, Spitzname pd  # Kap. 15

In [ ]:
import matplotlib.pyplot as plt  # Werkzeugkasten für Diagramme holen  # Kap. 20

In [ ]:
df_train = pd.read_csv("data/Train.csv")  # CSV-Datei einlesen und in df_train speichern  # Kap. 15
df_train.shape  # Anzahl Zeilen und Spalten anzeigen  # Kap. 15

**Ergebnis Milestone 0:**
- Trainingsdaten: 23.524 Zeilen (Personen), 13 Spalten.
- Testdaten: 10.086 Zeilen, 12 Spalten. Es fehlt nur `bank_account`, das sagen wir später vorher.
- Fast alle Spalten sind Text (Kategorien), nur `year`, `household_size` und `age_of_respondent` sind Zahlen.

## Milestone 1: EDA (erkundende Datenanalyse)

### 1. Überblick: Wie sehen die Daten aus?

In [ ]:
df_train.head()  # die ersten 5 Zeilen der Tabelle anzeigen  # Kap. 15

**Erste Zeilen:** Jede Zeile ist eine befragte Person, jede Spalte eine Information über sie (Land, Jahr, Wohnort, Handy, Alter, Bildung, Arbeit usw.).
Die Spalte `bank_account` ist unser **Ziel (target)**: Yes = hat ein Konto, No = hat kein Konto.

In [ ]:
df_train.info()  # Überblick: Spaltennamen, Datentypen, Anzahl gefüllter Werte  # Kap. 15

**Datentypen:** 10 der 13 Spalten sind Text (Kategorien), nur `year`, `household_size` und `age_of_respondent` sind Zahlen.
Für die Modelle müssen wir die Text-Spalten später in Zahlen umwandeln (Encoding).
`uniqueid` ist nur eine Personen-Nummer und hilft bei der Vorhersage nicht.

In [ ]:
df_test = pd.read_csv("data/Test.csv")  # Testdaten einlesen und in df_test speichern  # Kap. 15
df_test.shape  # Anzahl Zeilen und Spalten anzeigen  # Kap. 15

In [ ]:
df_train["bank_account"].value_counts()  # zählen, wie oft Yes und No vorkommt  # Kap. 15

### 2. Zielvariable: Wer hat ein Bankkonto?

In [ ]:
df_train["bank_account"].value_counts(normalize=True)  # dasselbe als Anteil (0 bis 1)  # Kap. 15

**Zielvariable:** 20.212 Personen (85,9 %) haben **kein** Bankkonto, nur 3.312 (14,1 %) haben eins.
Die Daten sind also **unausgewogen (imbalanced)**.
Ein Modell, das immer „No“ sagt, wäre schon zu 85,9 % richtig. Unser Modell muss besser sein.

In [ ]:
df_train["bank_account"].value_counts().plot(kind="bar")  # Balkendiagramm Yes/No  # Kap. 20
plt.title("Wer hat ein Bankkonto?")  # Überschrift  # Kap. 20
plt.xlabel("Bankkonto")  # Beschriftung unten  # Kap. 20
plt.ylabel("Anzahl Personen")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

**Diagramm:** Der Balken „No“ ist etwa sechsmal so hoch wie „Yes“. Das zeigt das Ungleichgewicht auf einen Blick.

### 3. Datenqualität: Fehlt etwas? Gibt es Doppelte?

In [ ]:
df_train.isna().sum()  # pro Spalte zählen, wie viele Werte fehlen  # Kap. 17

**Fehlende Werte:** In keiner Spalte fehlt ein Wert. Wir müssen nichts auffüllen oder löschen.

In [ ]:
df_train.duplicated().sum()  # zählen, wie viele Zeilen doppelt vorkommen  # neu

**Datenqualität:** Es gibt **keine fehlenden Werte** und **keine doppelten Zeilen**. Die Daten sind sauber.

### 4. Zusammenhänge: Was hängt mit einem Bankkonto zusammen?

In [ ]:
pd.crosstab(df_train["country"], df_train["bank_account"], normalize="index")  # Anteil Yes/No pro Land  # Kap. 17

**Bankkonto nach Land:** Kenia hat mit 25,1 % deutlich mehr Menschen mit Konto als Ruanda (11,5 %), Tansania (9,2 %) und Uganda (8,6 %).
Das Land ist also ein wichtiger Hinweis für das Modell.

In [ ]:
anteil_handy = pd.crosstab(df_train["cellphone_access"], df_train["bank_account"], normalize="index")  # Anteil Yes/No je Handy-Zugang  # Kap. 17
anteil_handy["Yes"].plot(kind="bar")  # nur den Anteil "Yes" als Balken zeigen  # Kap. 20
plt.title("Anteil mit Bankkonto nach Handy-Zugang")  # Überschrift  # Kap. 20
plt.xlabel("Zugang zu einem Handy?")  # Beschriftung unten  # Kap. 20
plt.ylabel("Anteil mit Konto")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

**Handy-Zugang:** Menschen **ohne** Zugang zu einem Handy haben fast nie ein Konto (ca. 1,7 %). Menschen **mit** Handy-Zugang haben etwa zehnmal so oft eins (ca. 18,4 %).
Handy-Zugang ist also ein **starker Hinweis** für das Modell.

In [ ]:
anteil_bildung = pd.crosstab(df_train["education_level"], df_train["bank_account"], normalize="index")  # Anteil Yes/No je Bildungsstufe  # Kap. 17
anteil_bildung["Yes"].sort_values().plot(kind="barh")  # Anteil "Yes" sortiert als liegende Balken  # neu
plt.title("Anteil mit Bankkonto nach Bildung")  # Überschrift  # Kap. 20
plt.xlabel("Anteil mit Konto")  # Beschriftung unten  # Kap. 20
plt.ylabel("Bildungsstufe")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

**Bildung:** Je mehr Bildung, desto häufiger ein Bankkonto.
- Ohne Schulbildung: nur ca. 4 %
- Grundschule: ca. 8,5 %
- Weiterführende Schule: ca. 23 %
- Hochschule: ca. 51 %
- Berufsausbildung: ca. 57 %, also sogar etwas mehr als bei Hochschule

Bildung ist damit einer der **stärksten Hinweise** für das Modell. Der Unterschied zwischen „ohne Schulbildung“ und „Berufsausbildung“ ist mehr als das Zehnfache.

In [ ]:
anteil_ort = pd.crosstab(df_train["location_type"], df_train["bank_account"], normalize="index")  # Anteil Yes/No für Stadt und Land  # Kap. 17
anteil_ort["Yes"].plot(kind="bar")  # Anteil "Yes" als Balken  # Kap. 20
plt.title("Anteil mit Bankkonto: Stadt oder Land")  # Überschrift  # Kap. 20
plt.xlabel("Wohnort (Rural = Land, Urban = Stadt)")  # Beschriftung unten  # Kap. 20
plt.ylabel("Anteil mit Konto")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

**Stadt oder Land:** In der Stadt (Urban) haben ca. 18 % ein Bankkonto, auf dem Land (Rural) ca. 12 %.
Der Unterschied ist vorhanden, aber **kleiner** als bei Bildung oder Handy-Zugang. Der Wohnort ist also ein **mittlerer Hinweis** für das Modell.

In [ ]:
df_train["age_of_respondent"].plot(kind="hist", bins=20)  # Altersverteilung in 20 Stufen  # Kap. 20
plt.title("Altersverteilung der Befragten")  # Überschrift  # Kap. 20
plt.xlabel("Alter in Jahren")  # Beschriftung unten  # Kap. 20
plt.ylabel("Anzahl Personen")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

In [ ]:
df_train.groupby("bank_account")["age_of_respondent"].mean()  # Durchschnittsalter je Gruppe Yes/No  # Kap. 17

**Alter:** Die Befragten sind zwischen 16 und 100 Jahre alt, die meisten zwischen ca. 20 und 40 Jahren.
Das Durchschnittsalter liegt ohne Konto bei ca. 38,7 Jahren und mit Konto bei ca. 39,6 Jahren, also weniger als 1 Jahr Unterschied.  
Das Alter allein ist damit nur ein **schwacher Hinweis** für das Modell.

In [ ]:
anteil_job = pd.crosstab(df_train["job_type"], df_train["bank_account"], normalize="index")  # Anteil Yes/No je Arbeitsart  # Kap. 17
anteil_job["Yes"].sort_values().plot(kind="barh")  # Anteil "Yes" sortiert als liegende Balken  # neu
plt.title("Anteil mit Bankkonto nach Art der Arbeit")  # Überschrift  # Kap. 20
plt.xlabel("Anteil mit Konto")  # Beschriftung unten  # Kap. 20
plt.ylabel("Art der Arbeit")  # Beschriftung links  # Kap. 20
plt.show()  # Diagramm anzeigen  # Kap. 20

**Art der Arbeit:** Fest Angestellte haben am häufigsten ein Konto: beim Staat (Formally employed Government) ca. 77 %, in Privatfirmen (Formally employed Private) ca. 54 %.  
Am seltensten haben Menschen in informeller Arbeit (Informally employed, ca. 8 %) oder ohne Einkommen (No Income, ca. 2 %) ein Konto.  
Die Art der Arbeit ist also ein **starker Hinweis** für das Modell.

### 5. Zusammenfassung der EDA

**Daten:** 23.524 Personen, 13 Spalten, keine fehlenden Werte, keine Duplikate.

**Zielvariable:** Nur 14,1 % haben ein Bankkonto, 85,9 % nicht. Die Daten sind **unausgewogen**.

**Was hängt mit einem Bankkonto zusammen?** (vom stärksten zum schwächsten Hinweis)

| Merkmal | Höchster Anteil mit Konto | Niedrigster Anteil mit Konto | Hinweis |
| --- | --- | --- | --- |
| Art der Arbeit | Staat: ca. 77 % | Ohne Einkommen: ca. 2 % | sehr stark |
| Bildung | Berufsausbildung: ca. 57 % | Ohne Schulbildung: ca. 4 % | sehr stark |
| Handy-Zugang | mit Handy: ca. 18,4 % | ohne Handy: ca. 1,7 % | stark |
| Land | Kenia: 25,1 % | Uganda: 8,6 % | mittel |
| Stadt oder Land | Stadt: ca. 18 % | Land: ca. 12 % | mittel |
| Alter | mit Konto: Ø 39,6 Jahre | ohne Konto: Ø 38,7 Jahre | schwach |

**Was bedeutet das für die Modelle?**
- Wegen des Ungleichgewichts muss unser Modell **besser als 85,9 %** richtig liegen. Das ist unsere Messlatte.
- Die meisten Spalten sind Text. Wir müssen sie vor dem Modell **in Zahlen umwandeln (Encoding)**.
- `uniqueid` ist nur eine Personen-Nummer und wird **nicht** als Hinweis verwendet.